# 01 - Exploratory Data Analysis
Olist Brazilian E-Commerce (CC BY-NC-SA 4.0). Requires one pipeline run (`python -m src.pipeline.run_pipeline`).

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd, numpy as np
import plotly.express as px
from config.database import read_sql
pd.set_option('display.max_columns', 50)
# NOTE: results reflect whatever data the pipeline loaded (DATA_SOURCE=kaggle -> real Olist, sample -> SYNTHETIC).

In [ ]:
read_sql('SELECT * FROM analytics.vw_data_window')

In [ ]:
read_sql('SELECT * FROM analytics.vw_executive_kpis').T

## Sales over time

In [ ]:
daily = read_sql('SELECT * FROM analytics.vw_daily_sales ORDER BY sales_date')
px.line(daily, x='sales_date', y='revenue', title='Daily revenue (BRL)')

In [ ]:
monthly = read_sql('SELECT * FROM analytics.vw_monthly_sales ORDER BY month')
px.bar(monthly, x='month', y='orders', title='Monthly orders')

## Category, product and payment mix

In [ ]:
cat = read_sql('SELECT product_category, SUM(units) units, SUM(revenue) revenue FROM analytics.vw_category_demand GROUP BY 1 ORDER BY 3 DESC LIMIT 15')
px.bar(cat, x='revenue', y='product_category', orientation='h', title='Top categories by revenue')

In [ ]:
read_sql('SELECT primary_payment_type, COUNT(*) orders, ROUND(AVG(max_installments),2) avg_installments FROM analytics.fact_orders GROUP BY 1 ORDER BY 2 DESC')

In [ ]:
read_sql('SELECT customer_state, COUNT(*) orders, ROUND(SUM(items_value)) revenue FROM analytics.fact_orders WHERE is_revenue_order GROUP BY 1 ORDER BY 2 DESC LIMIT 10')

## Customers
`customer_id` is unique per order, so repeat behaviour must use `customer_unique_id`.

In [ ]:
read_sql('SELECT * FROM analytics.vw_customer_kpis').T